# Quantum Data Encoding

Load classical data into quantum states three ways -- angle, amplitude, and IQP -- and see how each lays the same point out in a different feature space.

**Objectives:**
- Implement angle, amplitude, and IQP encodings with `lib.ml.feature_maps` and read out the exact state each prepares.
- Verify the defining property of amplitude encoding: the prepared state *is* the L2-normalized feature vector.
- Prove angle encoding of all-zeros leaves |0..0>, and that every encoding is deterministic (same input -> identical state).
- Visualize each encoding as a bar plot of |amplitude|^2 (and phase) from `state_vector()`.
- Compare qubit count and circuit depth across encodings to see the cost/expressivity trade-off.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

from lib.ml.feature_maps import angle_encoding, amplitude_encoding, iqp_encoding
from lib.utils.visualization import plot_histogram

# Local simulator: free, instant, exact state vectors.
device = LocalSimulator()
np.random.seed(0)  # determinism for the one sampling illustration below

# A small hardcoded toy dataset -- two 4D points, no sklearn / Iris loader.
# Kept in [0, 1] so the angles stay well below 2*pi (no Bloch-sphere aliasing).
X = np.array([
    [0.3, 0.7, 0.2, 0.9],
    [0.6, 0.1, 0.8, 0.4],
])
print("Toy dataset X (2 points, 4 features each):")
print(X)

## 1. Angle encoding: one feature per qubit

Angle encoding maps each feature to a `Ry` rotation: |phi(x)> = (X)_i Ry(x_i)|0>. One qubit per
feature, hardware-efficient. The feature space is the surface of `n` Bloch spheres. We start with a
single 2D point so the 4-amplitude state vector is easy to read.

In [ ]:
point2d = np.array([0.6, 0.9])
enc = angle_encoding(point2d)

print(enc)
print(f"\nqubit_count = {enc.qubit_count},  depth = {enc.depth}")

sv = enc.state_vector()
print("\nstate vector (big-endian, qubit 0 = leftmost bit):")
for i, amp in enumerate(sv):
    bits = format(i, f"0{enc.qubit_count}b")
    print(f"  |{bits}>  amp = {amp.real:+.4f}{amp.imag:+.4f}j   |amp|^2 = {abs(amp)**2:.4f}")

# Ry rotations produce purely real amplitudes here.
assert np.allclose(sv.imag, 0.0, atol=1e-12)

# <Z_0> for Ry(theta) on qubit 0 is exactly cos(theta). Compute it from the
# state vector: in big-endian, the sign is +1 when the top bit of i is 0.
probs = np.abs(sv) ** 2
n = enc.qubit_count
z0 = sum((1 if ((i >> (n - 1)) & 1) == 0 else -1) * probs[i] for i in range(len(sv)))
print(f"\n<Z_0> from state vector = {z0:.6f}   cos(0.6) = {np.cos(0.6):.6f}")
assert np.isclose(z0, np.cos(0.6), atol=1e-8)  # proves the encoding put x_0 on qubit 0's Y-rotation

### Angle encoding of all-zeros leaves |0..0>

`Ry(0)` is the identity, so encoding the zero vector must leave every qubit untouched. We span all
three qubits explicitly (qcsim measures all qubits at once, so the state vector length is
`2^(max_qubit+1)`).

In [ ]:
zeros_enc = angle_encoding(np.zeros(3))
sv_zero = zeros_enc.state_vector()

print(f"qubit_count = {zeros_enc.qubit_count}, state length = {len(sv_zero)}")
print("amplitudes:", np.round(sv_zero.real, 6))

expected = np.zeros(2 ** zeros_enc.qubit_count, dtype=complex)
expected[0] = 1.0  # |000>
assert np.allclose(sv_zero, expected, atol=1e-12)  # proves Ry(0) = identity on all qubits
print("\nangle_encoding(zeros) == |000>  : verified")

## 2. Amplitude encoding: pack N features into log2(N) qubits

Amplitude encoding writes the (L2-normalized) feature vector *directly* into the amplitudes:
|psi> = sum_i (f_i / ||f||) |i>. Exponentially compact -- a 4D point needs only 2 qubits -- but it
demands non-negative features and a length that is a power of 2, and costs O(N) gates to prepare.
The defining test: the prepared state vector equals the normalized features exactly.

In [ ]:
point4d = X[0]                       # [0.3, 0.7, 0.2, 0.9], all non-negative, length 4 = 2^2
normalized = point4d / np.linalg.norm(point4d)

enc_amp = amplitude_encoding(point4d)
sv_amp = enc_amp.state_vector()

print(f"qubit_count = {enc_amp.qubit_count},  depth = {enc_amp.depth}")
print("normalized features :", np.round(normalized, 6))
print("prepared amplitudes :", np.round(sv_amp.real, 6))

# THE defining property: state_vector() reproduces features / ||features|| exactly.
assert np.allclose(sv_amp, normalized, atol=1e-8)  # exact amplitude encoding
# And it is a valid (normalized) quantum state.
assert np.isclose(np.sum(np.abs(sv_amp) ** 2), 1.0, atol=1e-10)
print("\nstate_vector() == features / ||features||  : verified (atol=1e-8)")

### The contract amplitude encoding enforces

The Mottonen routine prepares *real, non-negative* amplitudes, so it rejects signed data, the zero
vector, and lengths that are not a power of 2. (For signed or odd-length data, reach for angle
encoding instead.) Watch it refuse each invalid input.

In [ ]:
bad_inputs = {
    "negative feature": np.array([0.3, -0.7, 0.2, 0.9]),
    "length not power of 2": np.array([0.3, 0.7, 0.2]),
    "zero vector": np.array([0.0, 0.0, 0.0, 0.0]),
}
for label, vec in bad_inputs.items():
    try:
        amplitude_encoding(vec)
        print(f"  {label}: NO error raised (unexpected)")
    except ValueError as err:
        print(f"  {label}: ValueError -> {err}")

## 3. IQP encoding: a structured, entangled feature space

IQP (Instantaneous Quantum Polynomial) encoding applies Hadamards, then single-qubit `Rz`
rotations, then `ZZ` interactions driven by feature *products* (`Rz` conjugated by `CNOT`s). It
builds an exponentially large, classically hard feature space -- the basis of quantum kernels with
potential advantage. We encode the same 2D point and look at the resulting probabilities and phases.

In [ ]:
enc_iqp = iqp_encoding(point2d)  # reps=2 by default

print(enc_iqp)
print(f"\nqubit_count = {enc_iqp.qubit_count},  depth = {enc_iqp.depth}")

sv_iqp = enc_iqp.state_vector()
probs_iqp = np.abs(sv_iqp) ** 2
print("\nbasis state   |amp|^2    phase (rad)")
for i, amp in enumerate(sv_iqp):
    bits = format(i, f"0{enc_iqp.qubit_count}b")
    phase = np.angle(amp) if abs(amp) > 1e-9 else 0.0
    print(f"  |{bits}>      {abs(amp)**2:.4f}     {phase:+.4f}")

# Hadamards spread amplitude across all basis states; ZZ terms imprint phases.
assert np.isclose(probs_iqp.sum(), 1.0, atol=1e-10)
assert np.all(probs_iqp > 1e-6)  # no basis state is left empty -- the feature space is dense
print("\nIQP spreads the point across all basis states with feature-product phases.")

## 4. Determinism: the same point always lands in the same state

Encoding is a fixed unitary built from the data -- no randomness. Encoding a point twice must give
bit-identical state vectors. We confirm this for all three maps (using the same data for each), which
is why every assert in this notebook is stable in CI.

In [ ]:
# angle and iqp take any length; amplitude needs non-negative, power-of-2 length.
assert np.array_equal(angle_encoding(point4d).state_vector(),
                      angle_encoding(point4d).state_vector())
assert np.array_equal(amplitude_encoding(point4d).state_vector(),
                      amplitude_encoding(point4d).state_vector())
assert np.array_equal(iqp_encoding(point4d).state_vector(),
                      iqp_encoding(point4d).state_vector())
print("All three encodings are deterministic: same input -> identical state vector.")

## 5. Visualize the states each encoding produces

The exact state vector lets us draw, for one shared 4D point, the |amplitude|^2 over basis states
for each encoding -- and the IQP phases, which carry the feature-product structure. (Angle and
amplitude encodings produce real amplitudes here, so their phases are 0.)

In [ ]:
shared = X[0]  # [0.3, 0.7, 0.2, 0.9]
encodings = {
    "angle": angle_encoding(shared),       # 4 qubits
    "amplitude": amplitude_encoding(shared),  # 2 qubits
    "iqp": iqp_encoding(shared),           # 4 qubits
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, circ) in zip(axes, encodings.items()):
    sv = circ.state_vector()
    nq = circ.qubit_count
    labels = [format(i, f"0{nq}b") for i in range(len(sv))]
    ax.bar(range(len(sv)), np.abs(sv) ** 2, color="#232f3e")
    ax.set_title(f"{name}  ({nq} qubits)")
    ax.set_xlabel("basis state |i>")
    ax.set_ylabel("|amplitude|^2")
    # Keep tick labels readable: only label when there are few basis states.
    if len(sv) <= 8:
        ax.set_xticks(range(len(sv)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
plt.suptitle("Same 4D point, three feature maps")
plt.tight_layout()
plt.show()

# Phases of the IQP state (the part angle/amplitude encodings leave at 0).
iqp_sv = encodings["iqp"].state_vector()
mask = np.abs(iqp_sv) > 1e-9
fig2, ax2 = plt.subplots(figsize=(8, 3.5))
idx = np.arange(len(iqp_sv))
ax2.bar(idx[mask], np.angle(iqp_sv)[mask], color="#ff9900")
ax2.set_title("IQP encoding: phase per basis state (rad)")
ax2.set_xlabel("basis state index i")
ax2.set_ylabel("phase")
plt.tight_layout()
plt.show()

### Now actually measure it

State vectors are an exact, simulator-only convenience. On hardware you only get samples. Encode the
point, run with shots, and histogram the empirical distribution -- it should track the angle
encoding's |amplitude|^2 bars above. (We seeded `np.random.seed(0)` in the setup cell for a stable
illustration; never base a tight assert on sampling.)

In [ ]:
result = device.run(angle_encoding(shared), shots=2000).result()
print("measurement counts:", dict(result.measurement_counts))
fig = plot_histogram(result.measurement_counts, title="Angle encoding -- 2000 shots")
plt.show()

## 6. Compare qubit count and circuit depth

The three encodings trade qubits against depth. Angle is the shallowest but needs one qubit per
feature. Amplitude is the most qubit-frugal (log2 N) but pays in depth to load the amplitudes. IQP
matches angle's qubit count but is by far the deepest, because of its all-pairs `ZZ` interactions
repeated over `reps`. We tabulate the exact numbers for the shared 4D point.

In [ ]:
rows = []
for name, circ in [
    ("angle", angle_encoding(shared)),
    ("amplitude", amplitude_encoding(shared)),
    ("iqp", iqp_encoding(shared)),
]:
    rows.append((name, circ.qubit_count, circ.depth))

print(f"{'encoding':<12}{'qubits':>8}{'depth':>8}")
print("-" * 28)
for name, q, d in rows:
    print(f"{name:<12}{q:>8}{d:>8}")

# Sanity checks on the trade-off (exact, deterministic):
q_angle, q_amp, q_iqp = rows[0][1], rows[1][1], rows[2][1]
d_angle, d_amp, d_iqp = rows[0][2], rows[1][2], rows[2][2]
assert q_angle == 4 and q_iqp == 4          # one qubit per feature
assert q_amp == 2                           # log2(4) qubits -- exponentially compact
assert d_iqp > d_amp > d_angle              # IQP deepest, angle shallowest
print("\nTrade-off confirmed: amplitude saves qubits, angle saves depth, IQP buys structure.")

## Exercises

In [ ]:
# Exercise 1: Re-scaling and aliasing -- mind the period.
# Ry(theta) has period 4*pi on the STATE (Ry(2*pi) = -I, Ry(4*pi) = I), while a
# Z-basis measurement probability cos^2(theta/2) has period 2*pi. So shifting a
# feature by a full turn aliases distinct inputs -- the danger of over-scaling.
#
# Encode [0.6, 0.9] as the reference, then:
#  (a) [0.6 + 4*np.pi, 0.9] -- the state vector should match the reference EXACTLY.
#  (b) [0.6 + 2*np.pi, 0.9] -- the state vector is NEGATED (a global -1 phase): it
#      looks different as a vector yet gives identical measurement statistics.
# Quantify both with np.max(np.abs(sv - sv_ref)) and explain what you see.
#
# TODO: build the three circuits and print np.round(.state_vector(), 4) for each.

# point_ref = np.array([0.6, 0.9])
# point_4pi = ...
# point_2pi = ...

In [ ]:
# Exercise 2: Fidelity overlap between two encoded points.
# The quantum-kernel building block is K(x1, x2) = |<phi(x1)|phi(x2)>|^2.
# For amplitude encoding you can compute it directly from state vectors:
# the overlap is np.vdot(sv1, sv2). Compute |<phi(X[0])|phi(X[1])>|^2 for the
# two toy points under amplitude_encoding, then under angle_encoding. Which
# encoding makes the two points look more similar? Why might that matter for a
# downstream classifier?
#
# TODO: sv1 = amplitude_encoding(X[0]).state_vector(); sv2 = ...
#       overlap = np.abs(np.vdot(sv1, sv2)) ** 2; print it for both encodings.


## Summary

- **Angle encoding** maps each feature to `Ry(x_i)` -- one qubit per feature, depth 1, real
  amplitudes; `<Z_0>` reads back `cos(x_0)`, and encoding zeros leaves |0..0>.
- **Amplitude encoding** writes the L2-normalized feature vector straight into the amplitudes:
  `state_vector() == features / ||features||` (exact). Exponentially compact (log2 N qubits) but it
  requires non-negative, power-of-2-length data and costs O(N) depth to prepare.
- **IQP encoding** spreads the point across all basis states with feature-product phases via
  Hadamards + `Rz` + `ZZ`, building a dense, classically hard feature space -- at the cost of being
  by far the deepest circuit.
- All three are deterministic fixed unitaries: same input -> identical state vector, which is what
  makes the asserts here exact and CI-stable.
- **The choice of encoding fixes the feature space the model ever gets to see** -- it is a modeling
  decision, not a formality.

**Next:** [`02-quantum-kernels.ipynb`](02-quantum-kernels.ipynb) -- turn these feature maps into a
fidelity kernel `K(x_i, x_j) = |<phi(x_i)|phi(x_j)>|^2` and classify a dataset that is linearly
inseparable in its raw coordinates.